# HCM AI Contest — Setup & Run
**Runtime:** A100 40GB (Colab Pro)

Chạy **5 cells** theo thứ tự. Cell cuối cùng xuất `submission/*.csv`.

In [15]:
from google.colab import drive
import os

# Ensure the mount point is empty
if os.path.exists('/content/drive') and os.listdir('/content/drive'):
    print("Thư mục /content/drive không trống. Vui lòng khởi động lại runtime hoặc dọn dẹp thư mục trước khi mount.")
else:
    drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
#@title 1. Install dependencies
#@markdown Clone repo + install requirements (~2 phút)
!git clone https://github.com/s4dz/aicv5.git /content/aicv5
%cd /content/aicv5
!pip install -r colab_requirements.txt -q
import urllib.request, os

# Fix missing BPE vocab file
target_dir = "/content/aicv5/tulip/open_clip/src/open_clip"
target_file = os.path.join(target_dir, "bpe_simple_vocab_16e6.txt.gz")
if not os.path.exists(target_file):
    os.makedirs(target_dir, exist_ok=True)
    url = "https://raw.githubusercontent.com/mlfoundations/open_clip/main/src/open_clip/bpe_simple_vocab_16e6.txt.gz"
    urllib.request.urlretrieve(url, target_file)
    print(f"Downloaded BPE vocab: {target_file}")

import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.0f} GB")

fatal: destination path '/content/aicv5' already exists and is not an empty directory.
/content/aicv5
GPU: NVIDIA A100-SXM4-80GB
VRAM: 79 GB


In [4]:
#@title 2. Load videos from Google Drive
#@markdown Videos are read from Drive (already mounted in Cell 0).
#@markdown Symlink to avoid copying 3GB.

import os, shutil
from pathlib import Path

PROJECT = Path('/content/aicv5')
VIDEO_DIR = PROJECT / 'data' / 'videos'
DRIVE_VIDEOS = Path('/content/drive/MyDrive/aicv5/videos')

# Check if videos exist on Drive
if DRIVE_VIDEOS.exists() and any(DRIVE_VIDEOS.glob('*.mp4')):
    print(f"Found videos on Drive: {DRIVE_VIDEOS}")
    # Symlink to project dir (avoids 3GB copy)
    if VIDEO_DIR.exists() and not VIDEO_DIR.is_symlink():
        shutil.rmtree(VIDEO_DIR)
    if not VIDEO_DIR.exists():
        os.symlink(str(DRIVE_VIDEOS), str(VIDEO_DIR))
        print(f"Symlinked: {VIDEO_DIR} -> {DRIVE_VIDEOS}")
    else:
        print(f"Symlink already exists: {VIDEO_DIR}")
else:
    print(f"No videos found on Drive at {DRIVE_VIDEOS}")
    print("Please upload videos to Drive/aicv5/videos/ first.")
    print("Falling back to ZIP download...")
    !pip install -q gdown
    import gdown, subprocess
    ZIP_ID = "1Ews9d2x5dwRMTbWlB9wfazorFmcDw4kL"
    VIDEO_DIR.mkdir(parents=True, exist_ok=True)
    zip_path = "/tmp/videos.zip"
    gdown.download(id=ZIP_ID, output=zip_path, quiet=False)
    subprocess.run(["unzip", "-q", "-o", zip_path, "-d", str(VIDEO_DIR)], check=True)
    # Fix: ZIP has files in video/ subfolder
    if (VIDEO_DIR / 'video').exists():
        !mv /content/aicv5/data/videos/video/*.mp4 /content/aicv5/data/videos/
        !rmdir /content/aicv5/data/videos/video

vid_count = len(list(VIDEO_DIR.glob('*.mp4')))
total = sum(f.stat().st_size for f in VIDEO_DIR.glob('*.mp4')) / 1024**3
print(f"Done: {vid_count} videos ({total:.1f} GB)")

Downloading...
From (original): https://drive.google.com/uc?id=1Ews9d2x5dwRMTbWlB9wfazorFmcDw4kL
From (redirected): https://drive.google.com/uc?id=1Ews9d2x5dwRMTbWlB9wfazorFmcDw4kL&confirm=t&uuid=4e2c5079-1929-46a6-87ad-7451ac623e79
To: /tmp/videos.zip
100%|██████████| 3.38G/3.38G [00:33<00:00, 101MB/s]


Extracting...
Done: 29 videos (3.1 GB)


In [8]:
!cd /content/aicv5 && find . -name "__pycache__" -exec rm -rf {} + && git pull

remote: Enumerating objects: 21, done.
remote: Counting objects: 100% (21/21), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 12 (delta 9), reused 12 (delta 9), pack-reused 0 (from 0)
Unpacking objects: 100% (12/12), 3.00 KiB | 767.00 KiB/s, done.
From https://github.com/s4dz/aicv5
   3cda685..1b218eb  main       -> origin/main
Updating 3cda685..1b218eb
Fast-forward
 colab_setup.ipynb                            |  56 +++++-------
 embedding/tulip_encoder.py                   | 123 +++++++++++++++++++--------
 tulip/open_clip/src/open_clip/transformer.py |   2 +-
 3 files changed, 110 insertions(+), 71 deletions(-)


In [6]:
#@title 3. Index video library (keyframe extraction + embeddings)
import urllib.request, os
import sys
from pathlib import Path

PROJECT = Path('/content/aicv5')
sys.path.insert(0, str(PROJECT))

KEYFRAMES = PROJECT / 'data' / 'keyframes'
STORE     = PROJECT / 'data' / 'persistent_store'

if STORE.exists() and any(STORE.iterdir()):
    print("Embeddings exist — skipping indexing.")
else:
    print("Running keyframe extraction + embeddings... (this may take a while)")
    from preprocessing.pipeline import PreprocessingPipeline
    pipeline = PreprocessingPipeline()
    video_files = sorted((PROJECT / 'data' / 'videos').glob('*.mp4'))
    print(f"Found {len(video_files)} videos")
    results = pipeline.keyframe_extractor.extract_batch(
    video_paths=[str(v) for v in video_files],
    output_dir=str(KEYFRAMES),
    save_keyframes=True,
    max_workers=16,  # 4 model copies song song
    )
    ok = sum(1 for r in results.values() if r and r[0].get('keyframes', 0) > 0)
    print(f"\nIndexed: {ok}/{len(video_files)} videos")

print(f"Keyframes: {KEYFRAMES}")
print(f"Store: {STORE}")

2026-08-18 09:10:08.262 | INFO     | preprocessing.keyframe_extractor:__init__:246 - KeyframeExtractor initialized: threshold=0.6, min_delta=5, sample_rate=5, batch_size=256
2026-08-18 09:10:08.263 | INFO     | preprocessing.pipeline:__init__:38 - Pipeline initialized: OCR=True, ASR=True
2026-08-18 09:10:08.264 | INFO     | preprocessing.keyframe_extractor:initialize:190 - Initializing model pool with 16 workers...


Running keyframe extraction + embeddings... (this may take a while)
Found 29 videos


open_clip_model.safetensors: reconstructing file:   0%|          |  0.00B / 1.71GB            

open_clip_model.safetensors: downloading bytes:           |  0.00B            

/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(
2026-08-18 09:10:16.933 | INFO     | preprocessing.keyframe_extractor:load:143 -   Worker 0: model loaded on cuda:0 (ViT-L-14, compiled)
2026-08-18 09:10:21.163 | INFO     | preprocessing.keyframe_extractor:load:143 -   Worker 1: model loaded on cuda:0 (ViT-L-14, compiled)
2026-08-18 09:10:25.522 | INFO     | preprocessing.keyframe_extractor:load:143 -   Worker 2: model loaded on cuda:0 (ViT-L-14, compiled)
2026-08-18 09:10:29.974 | INFO     | preprocessing.keyframe_extractor:load:143 -   Worker 3: model loaded on cuda:0 (ViT-L-14, compiled)
2026-08-18 09:10:34.400 | INFO     | preprocessing.keyframe_extractor:load:143 -   Worker 4: model loaded on cuda:0 (ViT-L-14, compiled)
2026-08-18 09:10:38.831 | INFO     | preprocessing.keyframe_extractor:load:143 -   Worker 5: model loaded o


Indexed: 0/29 videos
Keyframes: /content/aicv5/data/keyframes
Store: /content/aicv5/data/persistent_store


In [2]:
!find /content/aicv5 -name "__pycache__" -type d -exec rm -rf {} + 2>/dev/null
!find /content/aicv5 -name "*.pyc" -delete 2>/dev/null

In [10]:
#@title 3.5 Generate TULIP + SigLIP2 embeddings
#@markdown Encodes all extracted keyframes and saves to PersistentVectorStore.
#@markdown Skip if persistent_store already has entries.

import sys, time, os, re
from pathlib import Path
from collections import defaultdict
import numpy as np
from PIL import Image

PROJECT = Path('/content/aicv5')
sys.path.insert(0, str(PROJECT))

KEYFRAMES_DIR = PROJECT / 'data' / 'keyframes'
STORE_DIR     = PROJECT / 'data' / 'persistent_store'

# Check if store already populated
from retrieval.persistent_store import PersistentVectorStore
store = PersistentVectorStore(str(STORE_DIR))
if len(store._ids) > 0:
    print(f"Store already has {len(store._ids)} frames — skipping.")
else:
    # Load embedding engine (TULIP + SigLIP2)
    from embedding.hybrid_engine import HybridEmbeddingEngine
    engine = HybridEmbeddingEngine()
    print(f"Embedding engine loaded on {engine.device}")

    # Keyframes are flat: {video_name}_frame_{idx:06d}.jpg
    # Group by video_name
    _KF_RE = re.compile(r'^(.+?)_frame_(\d+)\.jpg$')
    grouped = defaultdict(list)  # video_name -> [(frame_idx, path)]
    for fp in sorted(KEYFRAMES_DIR.glob('*.jpg')):
        m = _KF_RE.match(fp.name)
        if m:
            grouped[m.group(1)].append((int(m.group(2)), fp))

    video_names = sorted(grouped.keys())
    print(f"Found {len(video_names)} videos, {sum(len(v) for v in grouped.values())} keyframes to encode")

    BATCH_SIZE = 64
    total_frames = 0
    t0 = time.time()

    for vi, vname in enumerate(video_names):
        frames = sorted(grouped[vname], key=lambda x: x[0])
        images = [Image.open(p).convert('RGB') for _, p in frames]
        frame_indices = [fi for fi, _ in frames]
        timestamps = [float(fi) for fi in frame_indices]  # frame_idx as proxy timestamp

        # Encode in batches
        all_clip = []
        all_siglip2 = []
        for bi in range(0, len(images), BATCH_SIZE):
            batch = images[bi:bi + BATCH_SIZE]
            embs = engine.encode_images_batch(batch)
            all_clip.append(embs['clip'])
            all_siglip2.append(embs['siglip2'])

        clip_embs = np.concatenate(all_clip, axis=0)
        siglip2_embs = np.concatenate(all_siglip2, axis=0)

        # Insert into store
        store.insert_frames(
            video_id=vname,
            frame_indices=frame_indices,
            clip_embeddings=clip_embs,
            siglip2_embeddings=siglip2_embs,
            timestamps=timestamps,
        )

        total_frames += len(images)
        elapsed = time.time() - t0
        fps = total_frames / elapsed if elapsed > 0 else 0
        print(f"  [{vi+1}/{len(video_names)}] {vname}: {len(images)} frames "
              f"({total_frames} total, {fps:.1f} fps)")

    elapsed = time.time() - t0
    print(f"\nDone: {total_frames} frames in {elapsed:.1f}s ({total_frames/elapsed:.1f} fps)")
    print(f"Store: {store.get_collection_stats()}")

2026-08-18 10:06:29.307 | INFO     | embedding.hybrid_engine:__init__:33 - Initializing Hybrid Embedding Engine...
2026-08-18 10:06:29.308 | INFO     | embedding.hybrid_engine:__init__:37 - Using TULIP encoder instead of CLIP
2026-08-18 10:06:29.308 | INFO     | embedding.tulip_encoder:__init__:59 - Initializing TULIP encoder on cuda:0
2026-08-18 10:06:36.705 | INFO     | embedding.tulip_encoder:__init__:74 - TULIP encoder loaded: ViT-L-14 (dim=768, fp16=True)
2026-08-18 10:06:36.706 | INFO     | embedding.siglip2_encoder:__init__:44 - Initializing SigLIP2 encoder on cuda:0
2026-08-18 10:06:55.418 | INFO     | embedding.siglip2_encoder:__init__:69 - SigLIP2 encoder loaded: ViT-SO400M-14-SigLIP2 (native_dim=1152, unified_dim=768)
2026-08-18 10:06:55.419 | INFO     | embedding.hybrid_engine:__init__:47 - Hybrid Embedding Engine initialized in 26.11s


Embedding engine loaded on cuda:0
Found 29 videos, 8740 keyframes to encode


2026-08-18 10:07:05.322 | DEBUG    | retrieval.persistent_store:_save_to_disk:77 - Saved 325 frames to persistent storage


  [1/29] L21_V001: 325 frames (325 total, 33.5 fps)


2026-08-18 10:07:14.436 | DEBUG    | retrieval.persistent_store:_save_to_disk:77 - Saved 629 frames to persistent storage


...

  [27/29] L21_V029: 303 frames (8135 total, 33.4 fps)


2026-08-18 10:11:07.913 | DEBUG    | retrieval.persistent_store:_save_to_disk:77 - Saved 8422 frames to persistent storage


  [28/29] L21_V030: 287 frames (8422 total, 33.4 fps)


2026-08-18 10:11:17.446 | DEBUG    | retrieval.persistent_store:_save_to_disk:77 - Saved 8740 frames to persistent storage


  [29/29] L21_V031: 318 frames (8740 total, 33.4 fps)

Done: 8740 frames in 261.8s (33.4 fps)
Store: {'name': 'persistent_store', 'num_entities': 8740, 'storage': '/content/aicv5/data/persistent_store'}


In [16]:
#@title 4. Start VLM server (vLLM + Qwen2.5-VL-32B)
#@markdown A100 80GB: chạy FP16 full, không cần quantize.
#@markdown Model download: ~30 phút first time, cached on Drive.

import os, subprocess, time

DRIVE_CACHE = '/content/drive/MyDrive/aicv5/checkpoints'
os.makedirs(DRIVE_CACHE, exist_ok=True)

# Start vLLM in background — FP16 for A100 80GB
VLM_CMD = (
    'python -m vllm.entrypoints.openai.api_server '
    '--model Qwen/Qwen2.5-VL-32B-Instruct '
    '--max-model-len 32768 '
    '--gpu-memory-utilization 0.90 '
    '--trust-remote-code '
    '--port 8080'
)
proc = subprocess.Popen(VLM_CMD, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)

# Wait for server to be ready
import httpx
print("Starting VLM server... ", end="")
for i in range(120):
    try:
        r = httpx.get('http://localhost:8080/health')
        if r.status_code == 200:
            print("READY")
            break
    except Exception:
        pass
    time.sleep(5)
    if i % 10 == 0:
        print(f"{i*5}s... ", end="")
else:
    print("TIMEOUT — check logs")

print(f"VLM endpoint: http://localhost:8080")
print(f"PID: {proc.pid}")

Starting VLM server... 0s... 50s... 100s... 150s... 200s... 250s... 300s... 350s... 400s... 450s... 500s... 550s... TIMEOUT — check logs
VLM endpoint: http://localhost:8080
PID: 34557


In [ ]:
#@title 5. Start API server (FastAPI + ngrok)
#@markdown Exposes the retrieval + VLM pipeline via public URL.

import subprocess, time, os
from pyngrok import ngrok

NGROK_TOKEN = ''  #@param {type:"string"}
if NGROK_TOKEN:
    ngrok.set_auth_token(NGROK_TOKEN)

# Start FastAPI in background
os.chdir('/content/aicv5')
api_proc = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'api.app:app', '--host', '0.0.0.0', '--port', '8000'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)

# Open ngrok tunnel
time.sleep(5)
public_url = ngrok.connect(8000).public_url
print(f"API server: {public_url}")
print(f"Health check: {public_url}/api/status")

# Verify
import httpx
try:
    r = httpx.get(f'{public_url}/api/status')
    print(f"Status: {r.json().get('status', 'unknown')}")
except Exception as e:
    print(f"Error: {e}")

In [ ]:
#@title 6. Run contest — generate submission CSVs
#@markdown Upload query .txt files to Drive/aicv5/queries/, then run.

import os, sys, csv
from pathlib import Path

os.chdir('/content/aicv5')
sys.path.insert(0, '.')

from retrieval.query_parser import detect_mode, parse_trake, split_qa, TKIS, QA, TRAKE
from retrieval.frame_sampler import sample_video_windows
from retrieval.trake_aligner import align_video_events, rank_videos
from retrieval.answer_expander import expand_rows, format_csv_line, SubmissionRow
from config import settings
import httpx, json

QUERIES_DIR = '/content/drive/MyDrive/aicv5/queries'
OUTPUT_DIR  = '/content/drive/MyDrive/aicv5/submission'
API_URL     = 'http://localhost:8000'  # or ngrok public URL

os.makedirs(OUTPUT_DIR, exist_ok=True)

def search_global(query_text, top_k=20):
    """Call API for global search."""
    # Encode query via API, then search
    r = httpx.post(f'{API_URL}/api/vqa/answer', json={'question': query_text}, timeout=120)
    return r.json()

queries = sorted(Path(QUERIES_DIR).glob('*.txt'))
print(f"Found {len(queries)} query files in {QUERIES_DIR}")

for q in queries:
    mode = detect_mode(q.name)
    text = q.read_text(encoding='utf-8').strip()
    csv_path = Path(OUTPUT_DIR) / f"{q.stem}.csv"
    print(f"  [{mode.upper():5s}] {q.name} → {csv_path.name}")

    # Pipeline dispatch — connect to your full pipeline here
    # For now, write placeholder
    lines = []
    # TODO: call _tkis_pipeline / _qa_pipeline / _trake_pipeline
    # from scripts.run_contest import _tkis_pipeline, _qa_pipeline, _trake_pipeline

    with open(csv_path, 'w', encoding='utf-8') as f:
        for line in lines:
            f.write(line + '\n')
    print(f"    → {len(lines)} rows")

print(f"\nDone. Files in {OUTPUT_DIR}/")
print("Download: right-click files → Download")